# 04 · KV-cache memory calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/04_kv_memory_calculator.ipynb) &nbsp; 📖 Concept deck: [KV cache](https://nilesh-salpe.github.io/visual-ai/inference/02-kv-cache/) &nbsp; ⏱️ Suggested time: **20 min**

**You will build:** the memory arithmetic every serving interview asks for: bytes per token, total cache size, how many concurrent sequences fit, and paging waste.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

print('no setup needed beyond numpy')

## Task 1: Bytes per token

Each layer stores one **K** and one **V** vector per kv head: $2 \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{bytes per element}$.

In [ ]:
def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, bytes_per_el=2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_kv_bytes_per_token(f):
    assert f(80, 8, 128, 2) == 327_680, "Llama-3-70B-shaped (80 layers, 8 kv heads, dim 128, BF16) should be 320 KB/token"
    assert f(32, 8, 128, 2) == 131_072, "Llama-3-8B-shaped should be 128 KB/token"
    assert f(80, 64, 128, 1) == 1_310_720, "64 kv heads (no GQA) in FP8"

check("kv_bytes_per_token", _test_kv_bytes_per_token, kv_bytes_per_token)

## Task 2: How many sequences fit?

Given GPU memory, model weight memory, a reserve fraction kept free for activations, and a per-sequence context length, return how many sequences' KV caches fit at once: $\lfloor ((1-\text{reserve}) \cdot \text{mem} - \text{weights}) / \text{bytes per sequence} \rfloor$, never below 0.

In [ ]:
def max_concurrent_seqs(gpu_bytes, weight_bytes, bytes_per_token, seq_len, reserve=0.1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_max_concurrent_seqs(f):
    GB = 1e9
    assert f(640 * GB, 140 * GB, 327_680, 8192) == 162, "8x80GB, 140GB weights, 70B-shaped KV, 8K context, 10% reserve -> 162"
    assert f(80 * GB, 140 * GB, 327_680, 8192) == 0, "weights don't fit -> 0, not negative"

check("max_concurrent_seqs", _test_max_concurrent_seqs, max_concurrent_seqs)

## Task 3: Paging vs contiguous reservation

Return a tuple `(contiguous_waste, paged_waste)`, each the fraction of **allocated** token slots that are unused.
- Contiguous: every sequence reserves `max_len` slots.
- Paged: every sequence allocates `ceil(len / block_size)` blocks of `block_size` slots. vLLM's default block size is 16.

In [ ]:
def paged_vs_contiguous_waste(seq_lens, max_len, block_size=16):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_paged_vs_contiguous_waste(f):
    c, p = f([100, 2000, 37], 2048, 16)
    assert abs(c - (1 - 2137 / 6144)) < 1e-9, f"contiguous waste wrong: {c}"
    assert abs(p - (1 - 2137 / ((7 + 125 + 3) * 16))) < 1e-9, f"paged waste wrong: {p}"

check("paged_vs_contiguous_waste", _test_paged_vs_contiguous_waste, paged_vs_contiguous_waste)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Bytes per token
def kv_bytes_per_token_solution(n_layers, n_kv_heads, head_dim, bytes_per_el=2):
    return 2 * n_layers * n_kv_heads * head_dim * bytes_per_el

check("kv_bytes_per_token (reference)", _test_kv_bytes_per_token, kv_bytes_per_token_solution)

In [ ]:
# @title Solution for Task 2: How many sequences fit?
def max_concurrent_seqs_solution(gpu_bytes, weight_bytes, bytes_per_token, seq_len, reserve=0.1):
    free = (1 - reserve) * gpu_bytes - weight_bytes
    return max(0, int(free // (bytes_per_token * seq_len)))

check("max_concurrent_seqs (reference)", _test_max_concurrent_seqs, max_concurrent_seqs_solution)

In [ ]:
# @title Solution for Task 3: Paging vs contiguous reservation
def paged_vs_contiguous_waste_solution(seq_lens, max_len, block_size=16):
    used = sum(seq_lens)
    contiguous = 1 - used / (max_len * len(seq_lens))
    paged = 1 - used / (sum(math.ceil(n / block_size) for n in seq_lens) * block_size)
    return contiguous, paged

check("paged_vs_contiguous_waste (reference)", _test_paged_vs_contiguous_waste, paged_vs_contiguous_waste_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
r = np.random.default_rng(0)
lens = list(r.integers(50, 2048, 64))
c, p = paged_vs_contiguous_waste_solution(lens, 2048, 16)
print(f"64 random requests: contiguous reservation wastes {c:.0%} of KV memory, paging wastes {p:.1%}")
bpt = kv_bytes_per_token_solution(80, 8, 128, 2)
for ctx in (2048, 8192, 32768, 131072):
    print(f"context {ctx:>6}: {bpt*ctx/2**30:6.2f} GiB of KV per sequence -> {max_concurrent_seqs_solution(640e9, 140e9, bpt, ctx)} sequences on 8x80 GB")

---
## Takeaways
- `KV bytes = 2 × layers × kv_heads × head_dim × bytes × tokens × batch`: memorize it.
- At long context the KV cache, not the weights, limits concurrency. That's why GQA, FP8 KV and paging matter.
- Paging wastes less than one block per sequence. Contiguous max-length reservation can waste most of the memory.